# Length distribution scaling debugging

It appears that there may be a bug with the length distribution scaling. Following scaling, segments of the genome do not normalize to the expected shape when compared to the replication profile deconvolution.

It's possible that segments of the genome are disproportionately influence the total length distribution curves. Check chrXII for example for omission.


In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [14]:
from src.timer import Timer

def compute_len_dist_for_replicate(replicate, timepoints=None, chroms=None):

    timer = Timer()

    from src.chromatin_metrics import ChromatinMetrics, fragment_lengths_definitions
    small_len, mid_len, nuc_len = fragment_lengths_definitions()
    chrom_metrics = ChromatinMetrics(small_len, mid_len, nuc_len)

    chrom_metrics.set_chrom(chrom=1, replicate=replicate)
    
    if timepoints is None:
        timepoints = chrom_metrics.chrom_reads['sample'].unique()
    timepoint_counts = {}
    
    print(f"Computing the length distribution for replicate {replicate}")

    timepoint_chrom_counts = {}
    if chroms is None:
        chroms = np.arange(1, 17)
    
    for timepoint in timepoints:
        cumulative_counts = None
        
        print(f"   t={timepoint}. For Chr: ", end=" ")
        
        chrom_counts = {}
        for chrom in chroms:

            print(f"{chrom}", end=", ")
            chrom_metrics.set_chrom(chrom=chrom, replicate=replicate)
            lens, counts = chrom_metrics.compute_length_hist(timepoint)
            chrom_counts[chrom] = counts
            
            if cumulative_counts is None:
                cumulative_counts = counts
            else:
                cumulative_counts = cumulative_counts + counts

        timepoint_chrom_counts[timepoint] = chrom_counts
        timepoint_counts[timepoint] = cumulative_counts
        timepoints_counts_df = pd.DataFrame(timepoint_counts)
        print()

    print(f"Completed {timer.get_time()}")

    return chrom_metrics, timepoints_counts_df, timepoint_chrom_counts

In [172]:
from src.global_config import GlobalConstants

timepoints = GlobalConstants.CHROM_WT1_TIMEPOINTS
chroms = range(1, 17)

chrom_metrics, timepoints_counts_df, timepoint_chrom_counts = \
    compute_len_dist_for_replicate(1, timepoints=timepoints, chroms=chroms)


Computing the length distribution for replicate 1
   t=0. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=10. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=20. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=30. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=40. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=50. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=60. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=70. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=80. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=90. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=100. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=110. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=120. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10

In [234]:

total_df = pd.DataFrame()

for timepoint in timepoints:
    timepoint_len_counts_df = pd.DataFrame(timepoint_chrom_counts[timepoint]).T
    timepoint_len_counts_df.index.name = 'chrom'
    timepoint_len_counts_df = timepoint_len_counts_df.reset_index()
    timepoint_len_counts_df['timepoint'] = timepoint
    timepoint_len_counts_df = timepoint_len_counts_df.set_index(['timepoint', 'chrom'])
    total_df = pd.concat([total_df, timepoint_len_counts_df])
    

In [235]:
total_df

0    1    2    3    4    5    6    7    8    9    ...   241  \
timepoint chrom                                                    ...         
0         1        0    0    0    0    0    0    0    0    0    0  ...   864   
          2        0    0    0    0    0    0    0    0    0    0  ...  4072   
          3        0    0    0    0    0    0    0    0    0    0  ...  1594   
          4        0    0    0    0    0    0    0    0    0    0  ...  7688   
          5        0    0    0    0    0    0    0    0    0    0  ...  3068   
...              ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   ...   
150       12       0    0    0    0    0    0    0    0    0    0  ...  4827   
          13       0    0    0    0    0    0    0    0    0    0  ...  4336   
          14       0    0    0    0    0    0    0    0    0    0  ...  3331   
          15       0    0    0    0    0    0    0    0    0    0  ...  5254   
          16       0    0    0    0    0    0    0    0    0    0  ...  4570   

                  242   243   244   245   246   247   248   249   250  
timepoint chrom                                                        
0         1       871   837   830   840   794   830   849   842   852  
          2      3900  3930  3999  3845  3915  3803  3779  3785  3795  
          3      1513  1539  1684  1570  1623  1670  1709  1591  1644  
          4      7496  7418  7399  7383  7324  7060  7226  7076  6856  
          5      2959  2933  2917  2898  3011  2884  2925  2743  2932  
...               ...   ...   ...   ...   ...   ...   ...   ...   ...  
150       12     4837  4691  4677  4613  4631  4706  4666  4627  4587  
          13     4337  4181  4307  4156  4302  4302  4040  4172  4088  
          14     3445  3522  3514  3424  3372  3415  3324  3337  3342  
          15     5088  5114  5000  5001  4976  4930  5070  4736  4837  
          16     4406  4381  4410  4448  4314  4240  4335  4214  4233  

[256 rows x 251 columns]